# Suite CONF — Cross-implementation conformance

Every implementation builds the same corpus (`mbse/Schemas/Conformance/Corpus.py`, mirrored statement for statement in
TypeScript) and commits each case's store, written as a log of mutations (its initial snapshot), under
`conformance/<implementation>/`. This suite checks this implementation's
files are current, that its JSON is byte-identical to every other implementation's, and that it reads every other
implementation's JSON and YAML back into a store that writes the same log. Regenerate with `uv run python -m mbse.Schemas.Conformance.write`.

In [ ]:
from pathlib import Path

import yaml

from mbse.Schemas.Conformance.Corpus import CASES, build
from mbse.Schemas.Conformance.write import render
from mbse.Schemas.Framework import Builders, JSON, Plain, Proxies, Validators, YAML


ROOT = Path("../../conformance")
OWN = "python3"
OTHERS = sorted(p.name for p in ROOT.iterdir() if p.is_dir() and p.name != OWN)
assert OTHERS == ["typescript5"], OTHERS

corpus = build()  # each case's store, and its data, which a store holds weakly
snapshots = {case: Plain.ToPlain(store).Snapshot() for case, (store, _) in corpus.items()}

## CONF-01 · This implementation's committed files are current

In [ ]:
for name, text in render(corpus).items():
    assert (ROOT / OWN / name).read_text(encoding="utf-8") == text, f"{name} is stale: regenerate the corpus"

## CONF-02 · Every implementation has the same cases, and JSON is byte-identical across implementations

In [ ]:
for name in [OWN, *OTHERS]:
    assert sorted(p.name for p in (ROOT / name).iterdir()) == sorted(f"{c}.{e}" for c in CASES for e in ("json", "yaml")), name
for other in OTHERS:
    for case in CASES:
        mine = (ROOT / OWN / f"{case}.json").read_bytes()
        theirs = (ROOT / other / f"{case}.json").read_bytes()
        assert mine == theirs, f"{case}.json differs from {other}"

## CONF-03 · Every implementation's YAML reads back to exactly this implementation's log, also under YAML 1.1

In [ ]:
for other in [OWN, *OTHERS]:
    for case in CASES:
        text = (ROOT / other / f"{case}.yaml").read_text(encoding="utf-8")
        assert YAML.loads(text) == snapshots[case], f"{other}/{case}.yaml"
        assert yaml.safe_load(text) == snapshots[case], f"{other}/{case}.yaml under YAML 1.1"

## CONF-04 · Every implementation's JSON and YAML read into another store, which writes the same log, its data valid

In [ ]:
for other in [OWN, *OTHERS]:
    for case in CASES:
        for suffix, decode in [("json", lambda s: JSON.FromJSON(s).Snapshot), ("yaml", lambda s: YAML.FromYAML(s).Snapshot)]:
            fresh = Builders.Store()
            made = decode(fresh)((ROOT / other / f"{case}.{suffix}").read_bytes())
            assert Plain.ToPlain(fresh).Snapshot() == snapshots[case], f"{other}/{case}.{suffix}"
            for value in made:
                if isinstance(value, Proxies._ObjectData):
                    assert Validators.Validate(fresh).Reachable(value.as_schema(), value) == [], (other, case, suffix)